In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
%pip install yfinance lightgbm scikit-learn requests --quiet

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 7, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [24]:
import datetime

TELEGRAM_TOKEN   = "8659001033:AAG9wyr_CdGgL0hvY6C1LQ53jrr5IRFvqFA"
TELEGRAM_CHAT_ID = "-5200409875"

TODAY = datetime.date.today().isoformat()

TICKERS = {
    "nasdaq_price_usd":            "^IXIC",
    "sp500_price_usd":             "^GSPC",
    "dowjones_price_usd":          "^DJI",
    "china_shanghai_price_usd":    "000001.SS",
    "hongkong_hongkong_price_usd": "^HSI",
    "uk_london_price_usd":         "^FTSE",
    "japan_tokyo_price_usd":       "^N225",
    "egx30_price_egp":             "^CASE30",
    "brent_oil_price_usd":         "BZ=F",
    "gold_price_usd":              "GC=F",
}

USD_STOCK_MARKETS = [
    ("NASDAQ",    "nasdaq_price_usd",            "cascade_NASDAQ_top_model.pkl",    "USD"),
    ("S&P 500",   "sp500_price_usd",             "cascade_SandP_500_top_model.pkl","USD"),
    ("Dow Jones", "dowjones_price_usd",           "cascade_Dow_Jones_top_model.pkl", "USD"),
    ("Shanghai",  "china_shanghai_price_usd",     "cascade_Shanghai_top_model.pkl",  "USD"),
    ("Hong Kong", "hongkong_hongkong_price_usd",  "cascade_Hong_Kong_top_model.pkl", "USD"),
    ("London",    "uk_london_price_usd",          "cascade_London_top_model.pkl",    "USD"),
    ("Tokyo",     "japan_tokyo_price_usd",        "cascade_Tokyo_top_model.pkl",     "USD"),
]

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 31, Finished, Available, Finished, False)

In [25]:
import yfinance as yf
import numpy as np

today_prices = {}
for col, ticker in TICKERS.items():
    try:
        price = yf.Ticker(ticker).fast_info.last_price
        today_prices[col] = round(float(price), 4)
        print(f"  {col:<40} {price:>12,.4f}")
    except Exception as e:
        today_prices[col] = 0.0
        print(f"  [WARN] {col}: {e}")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 32, Finished, Available, Finished, False)

  nasdaq_price_usd                          26,247.0801
  sp500_price_usd                            7,398.9302
  dowjones_price_usd                        49,609.1602
  china_shanghai_price_usd                   4,179.9531
  hongkong_hongkong_price_usd               26,393.7109
  uk_london_price_usd                       10,233.0996
  japan_tokyo_price_usd                     62,713.6484
  egx30_price_egp                           54,628.6992
  brent_oil_price_usd                          101.2900
  gold_price_usd                             4,730.7002


In [26]:
import pickle
import os
from notebookutils import mssparkutils

MODEL_BASE = "Files/models"

def load_model(filename):
    """
    Copy pkl from Lakehouse Files to local /tmp then unpickle
    """

    src = f"{MODEL_BASE}/{filename}"
    local = f"/tmp/{filename}"

    # remove existing local file first
    if os.path.exists(local):
        os.remove(local)

    # copy from lakehouse to local temp
    mssparkutils.fs.cp(src, f"file:{local}")

    # load pickle
    with open(local, "rb") as f:
        return pickle.load(f)


print("Loading models...")

models = {}

for name, col, pkl_file, ccy in USD_STOCK_MARKETS:

    models[name] = load_model(pkl_file)

    print(f"  ✓ {name}")


egx30_model = load_model("cascade_EGX30_top_model.pkl")
print("  ✓ EGX30")

oil_model = load_model("cascade_oil_top_model.pkl")
print("  ✓ Oil")

gold_model = load_model("cascade_gold__top_model.pkl")
print("  ✓ Gold")

print("All models loaded.")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 33, Finished, Available, Finished, False)

Loading models...
  ✓ NASDAQ
  ✓ S&P 500
  ✓ Dow Jones
  ✓ Shanghai
  ✓ Hong Kong
  ✓ London
  ✓ Tokyo
  ✓ EGX30
  ✓ Oil
  ✓ Gold
All models loaded.


In [27]:
import pandas as pd

def make_row(model, context_dict):
    """
    Build a 1-row DataFrame with exactly the features the model expects.
    Uses context_dict for values, fills missing features with 0.
    """
    row = {}
    for feat in model.feature_name_:
        row[feat] = context_dict.get(feat, 0.0)
    return pd.DataFrame([row])[model.feature_name_]

def predict_tomorrow(model, context_dict, today_price):
    """Predict log-return then reconstruct price."""
    X       = make_row(model, context_dict)
    ret     = model.predict(X)[0]
    return today_price * np.exp(ret)

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 34, Finished, Available, Finished, False)

In [28]:
stock_predictions = {}   # col_name → predicted tomorrow price

print("─" * 50)
print("STAGE 1 — Stock Markets")
print("─" * 50)

for name, col, pkl_file, ccy in USD_STOCK_MARKETS:
    model      = models[name]
    pred_price = predict_tomorrow(model, today_prices, today_prices[col])
    stock_predictions[col] = pred_price
    change_pct = (pred_price / today_prices[col] - 1) * 100
    print(f"  {name:<14}  Today: {today_prices[col]:>10,.2f}  →  Tomorrow: {pred_price:>10,.2f}  ({change_pct:+.2f}%)")

# EGX30 standalone
egx30_pred = predict_tomorrow(egx30_model, today_prices, today_prices["egx30_price_egp"])
print(f"  {'EGX30':<14}  Today: {today_prices['egx30_price_egp']:>10,.2f}  →  Tomorrow: {egx30_pred:>10,.2f}  EGP")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 35, Finished, Available, Finished, False)

──────────────────────────────────────────────────
STAGE 1 — Stock Markets
──────────────────────────────────────────────────
  NASDAQ          Today:  26,247.08  →  Tomorrow:  26,249.38  (+0.01%)
  S&P 500         Today:   7,398.93  →  Tomorrow:   7,399.80  (+0.01%)
  Dow Jones       Today:  49,609.16  →  Tomorrow:  49,612.28  (+0.01%)
  Shanghai        Today:   4,179.95  →  Tomorrow:   4,179.02  (-0.02%)
  Hong Kong       Today:  26,393.71  →  Tomorrow:  26,396.31  (+0.01%)
  London          Today:  10,233.10  →  Tomorrow:  10,233.70  (+0.01%)
  Tokyo           Today:  62,713.65  →  Tomorrow:  62,708.95  (-0.01%)
  EGX30           Today:  54,628.70  →  Tomorrow:  54,633.50  EGP


In [29]:
print("─" * 50)
print("STAGE 2 — Brent Oil")
print("─" * 50)

oil_context = {**today_prices, **stock_predictions}
oil_pred    = predict_tomorrow(oil_model, oil_context, today_prices["brent_oil_price_usd"])
oil_change  = (oil_pred / today_prices["brent_oil_price_usd"] - 1) * 100
print(f"  Today: {today_prices['brent_oil_price_usd']:,.4f}  →  Tomorrow: {oil_pred:,.4f}  ({oil_change:+.2f}%)")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 36, Finished, Available, Finished, False)

──────────────────────────────────────────────────
STAGE 2 — Brent Oil
──────────────────────────────────────────────────
  Today: 101.2900  →  Tomorrow: 101.2787  (-0.01%)


In [30]:
print("─" * 50)
print("STAGE 3 — Gold")
print("─" * 50)

gold_context = {
    **today_prices,
    **stock_predictions,
    "brent_oil_price_usd": oil_pred,      # predicted oil, not actual
}
gold_pred   = predict_tomorrow(gold_model, gold_context, today_prices["gold_price_usd"])
gold_change = (gold_pred / today_prices["gold_price_usd"] - 1) * 100
print(f"  Today: {today_prices['gold_price_usd']:,.4f}  →  Tomorrow: {gold_pred:,.4f}  ({gold_change:+.2f}%)")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 37, Finished, Available, Finished, False)

──────────────────────────────────────────────────
STAGE 3 — Gold
──────────────────────────────────────────────────
  Today: 4,730.7002  →  Tomorrow: 4,731.8692  (+0.02%)


In [31]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

pred_rows = []

# ── Global markets ────────────────────────────────────────────────────────────
for name, col, _, ccy in USD_STOCK_MARKETS:

    pred_rows.append({
        "prediction_date": TODAY,
        "label": name,
        "col_name": col,
        "currency": ccy,
        "today_price": today_prices[col],
        "predicted_tomorrow": stock_predictions[col],
    })


# ── EGX30 ─────────────────────────────────────────────────────────────────────
pred_rows.append({
    "prediction_date": TODAY,
    "label": "EGX30",
    "col_name": "egx30_price_egp",
    "currency": "EGP",
    "today_price": today_prices["egx30_price_egp"],
    "predicted_tomorrow": egx30_pred
})


# ── Brent Oil ─────────────────────────────────────────────────────────────────
pred_rows.append({
    "prediction_date": TODAY,
    "label": "Brent Oil",
    "col_name": "brent_oil_price_usd",
    "currency": "USD",
    "today_price": today_prices["brent_oil_price_usd"],
    "predicted_tomorrow": oil_pred
})


# ── Gold ──────────────────────────────────────────────────────────────────────
pred_rows.append({
    "prediction_date": TODAY,
    "label": "Gold",
    "col_name": "gold_price_usd",
    "currency": "USD",
    "today_price": today_prices["gold_price_usd"],
    "predicted_tomorrow": gold_pred
})


# ── Create Spark DF ───────────────────────────────────────────────────────────
df_spark = spark.createDataFrame(pd.DataFrame(pred_rows))


# ── Save Delta Table ──────────────────────────────────────────────────────────
df_spark.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("DailyPredictions")


# ── Save CSV backup ───────────────────────────────────────────────────────────
df_spark.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"Files/data/predictions_{TODAY}.csv")


print(f"✓ Predictions saved — {len(pred_rows)} assets")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 38, Finished, Available, Finished, False)

✓ Predictions saved — 10 assets


In [32]:
import requests

def send_telegram(token, chat_id, message):
    url  = f"https://api.telegram.org/bot{token}/sendMessage"
    resp = requests.post(url, json={"chat_id": chat_id, "text": message,
                                    "parse_mode": "Markdown"}, timeout=10)
    return resp.status_code == 200

# Build the full daily report
lines = [
    f"🔮 *Daily Market Predictions*",
    f"📅 As of {TODAY} — Predictions for tomorrow\n",
    "*Stock Markets:*",
]
for name, col, _, ccy in USD_STOCK_MARKETS:
    t  = today_prices[col]
    p  = stock_predictions[col]
    ch = (p / t - 1) * 100
    arrow = "📈" if ch > 0 else "📉"
    lines.append(f"  {arrow} *{name}*: `{t:,.2f}` → `{p:,.2f}` ({ch:+.2f}%) {ccy}")

egx30_ch = (egx30_pred / today_prices["egx30_price_egp"] - 1) * 100
egx30_arrow = "📈" if egx30_ch > 0 else "📉"
lines.append(f"  {egx30_arrow} *EGX30*: `{today_prices['egx30_price_egp']:,.2f}` → `{egx30_pred:,.2f}` ({egx30_ch:+.2f}%) EGP")

oil_arrow  = "📈" if oil_change > 0 else "📉"
gold_arrow = "📈" if gold_change > 0 else "📉"

lines += [
    "\n*Commodities:*",
    f"  {oil_arrow} *Brent Oil*: `{today_prices['brent_oil_price_usd']:,.4f}` → `{oil_pred:,.4f}` ({oil_change:+.2f}%) USD",
    f"  {gold_arrow} *Gold*:      `{today_prices['gold_price_usd']:,.4f}` → `{gold_pred:,.4f}` ({gold_change:+.2f}%) USD",
    "\n_Cascade: Stocks → Oil → Gold_",
]

message = "\n".join(lines)

tg_ok = send_telegram(TELEGRAM_TOKEN, TELEGRAM_CHAT_ID, message)

print(f"✓ Telegram: {'sent' if tg_ok else 'FAILED'}")

notebookutils.notebook.exit("success")

StatementMeta(, cb5d39d2-6ae8-4a4b-8b42-71877be300b7, 39, Finished, Available, Finished, False)

✓ Telegram: sent
ExitValue: success